# OT-2: Gas Generator Combustor Sizing

In [225]:
import numpy as np
import pandas as pd

import yaml

from datetime import datetime
import textwrap

from turborocket.components.combustion.devices import GasGenerator

from turborocket.fluids.fluids import IncompressibleFluid, CoolPropFluid
from turborocket.components.combustion.injector import InjectorMethods
from turborocket.components.combustion.devices import PropellantTypes

import handcalcs.render

Author: Elias Aoubala

Date: 22/02/2026

## 1 - Background

This document contains the authors sizing analysis conducted for the `OT-2: Man-Ray and The Dirty Bubble` turbopump. The main aim of this document is to conduct the sizing of the gas-generator, based on user specified design characteristics.

The user creates a `gg_in.yaml` file which contains the high level functional specifications of the gas-generator, which is read-in and whos results are presented to the user.

Based on these inputs, sizing of the gas-generator is performed using the in-house developed `turboRocket` python package.

Resulting combustion conditions and dimensions of the gas generator are then exported out to the user accordingly.

## 2 - High Level Functional Parmaeters (as loaded from yaml file)

Here, we can import our high level gas-generator sizing file, to get all the high level key inputs from the user. We just need to state the directory where the files are located under, which in this case is the `config` folder.

In [226]:
input_directory = "config"

We can now load in our input parameters into the gas-generator accordingly.

In [227]:
with open(f"{input_directory}/gg_in.yaml") as f:

    gg_inputs = yaml.safe_load(f)

We can now pull out our main gas generator chamber and injector objects.

In [228]:
chamber = gg_inputs["gas_generator"]["chamber"]

fu_injector = gg_inputs["gas_generator"]["injector"]["fuel"]

ox_injector = gg_inputs["gas_generator"]["injector"]["oxidiser"]

## 3 - Instantiating our Chamber - Injector Assembly

Now that we have loaded up our parameters, we can instantiate our chamber injector assembly.

In [229]:
gas_gen = GasGenerator(name = "Gas Generator",
                       fu_injector = fu_injector["name"] , 
                       ox_injector = ox_injector["name"])

We can setup our dictionary of propellants cantera alias;

In [230]:
prop_alias = {
    "oxidiser": ox_injector["ctr_alias"],
    "fuel": fu_injector["ctr_alias"]
}

We can now setup the cantera combustion.

In [231]:
gas_gen.setup_cantera(cantera_alias = prop_alias, combustion_file=chamber["ctr_file"])

## 4 - Sizing our Combustin Chamber

We can size the combustion chamber based on the chamber pressure, mass flow and mixture ratio specified,.

In [232]:
gas_gen.size_chamber(MR = chamber["MR"],
                     fuel = fu_injector["ctr_alias"],
                     oxidiser = ox_injector["ctr_alias"],
                     p_cc = chamber["p_cc"],
                     m_dot = chamber["m_dot_t"],
                     eta_c = chamber["eta_c"])

ERROR: FAILURE its = 1001!
ERROR: FAILURE its = 1001!
ERROR: FAILURE its = 1001!
ERROR: FAILURE its = 1001!
ERROR: FAILURE its = 1001!
ERROR: FAILURE its = 1001!
ERROR: FAILURE its = 1001!
ERROR: FAILURE its = 1001!
ERROR: FAILURE its = 1001!
ERROR: FAILURE its = 1001!


## 5 - Sizing the Injector Elements

We can now size the injector elements for both oxidiser and fuel.

### 5.1 - Fuel Injector

We firstly instantiate our fuel inlet Object

In [233]:
fu_params = fu_injector["inlet_params"]

m_dot_f = chamber["m_dot_t"]  / (1 + chamber["MR"])

if fu_injector["fluid_model"] == "IncompressibleFluid":
    fuel_inlet = IncompressibleFluid(name = fu_injector["name"],
                                     p = fu_params["p"],
                                     t = fu_params["t"],
                                     rho = fu_params["rho"]
                                     )
    
elif fu_injector["fluid_model"] == "CoolProp":
    fuel_inlet = CoolPropFluid(name = fu_injector["name"],
                               p = fu_params["p"],
                               t = fu_params["t"])
    
else:
    raise ValueError(f"Fluid model not implemented yet: {fu_injector["fluid_model"]}")

In [234]:
gas_gen.size_injector(inlet = fuel_inlet, 
                      p_cc = chamber["p_cc"],
                      m_dot = m_dot_f,
                      prop_flag = PropellantTypes.fuel,
                      inj_type = InjectorMethods(fu_injector["inj_model"]))

### 5.2 - Oxidiser Injector

We firstly instantiate our oxidiser inlet Object

In [235]:
ox_params = ox_injector["inlet_params"]

m_dot_f = chamber["m_dot_t"] * chamber["MR"] / (1 + chamber["MR"])

if ox_injector["fluid_model"] == "IncompressibleFluid":
    ox_inlet = IncompressibleFluid(name = ox_injector["name"],
                                     p = ox_params["p"],
                                     t = ox_params["t"],
                                     rho = ox_params["rho"]
                                     )
    
elif ox_injector["fluid_model"] == "CoolProp":
    ox_inlet = CoolPropFluid(name = ox_injector["name"],
                               p = ox_params["p"],
                               t = ox_params["t"])
    
else:
    raise ValueError(f"Fluid model not implemented yet: {ox_injector["fluid_model"]}")

In [236]:
gas_gen.size_injector(inlet = ox_inlet, 
                      p_cc = chamber["p_cc"],
                      m_dot = m_dot_f,
                      prop_flag = PropellantTypes.oxidiser,
                      inj_type = InjectorMethods(ox_injector["inj_model"]))

## 6 - Simulating Performance and extacting geomtries

We can now evaluate for the gas-generator conditions

In [237]:
performance = gas_gen.evaluate_condition(inlet_fuel=fuel_inlet, inlet_oxidiser=ox_inlet, eta_c=chamber["eta_c"])

geom = gas_gen.geometry()

ox: 0.04359522249276918
fu: 0.07620634755325442
ERROR: FAILURE its = 1001!
ox: 0.043464240056029196
fu: 0.07597738454937077
ERROR: FAILURE its = 1001!
ox: 0.04702044217905243
fu: 0.0912096366459311
ERROR: FAILURE its = 1001!
ERROR: FAILURE its = 1001!
ERROR: FAILURE its = 1001!
ERROR: FAILURE its = 1001!
ERROR: FAILURE its = 1001!
ox: 0.0467105646712092
fu: 0.09306496637082701
ERROR: FAILURE its = 1001!
ERROR: FAILURE its = 1001!
ERROR: FAILURE its = 1001!
ERROR: FAILURE its = 1001!
ERROR: FAILURE its = 1001!
ox: 0.04666669243376167
fu: 0.09333317525010332
ERROR: FAILURE its = 1001!
ERROR: FAILURE its = 1001!
ERROR: FAILURE its = 1001!
ERROR: FAILURE its = 1001!
ERROR: FAILURE its = 1001!
ERROR: FAILURE its = 1001!
ERROR: FAILURE its = 1001!
ERROR: FAILURE its = 1001!
ERROR: FAILURE its = 1001!
ERROR: FAILURE its = 1001!


## 7 - Getting our Orifice Sizes

We can get the orifice size:

In [238]:
A_f = geom["cda_fu"] / fu_injector["cd"]
A_o = geom["cda_ox"] / ox_injector["cd"]

A_f_orf = A_f / fu_injector["N"]
A_o_orf = A_o / ox_injector["N"]

From here, we can get the diameter:

In [239]:
D_f_orf = (A_f_orf/np.pi)**(1/2) * 2
D_o_orf = (A_o_orf/np.pi)**(1/2) * 2

We can then print

In [240]:
print(f"Fuel Diameter: {D_f_orf*1e3:.2f} mm")
print(f"Oxidiser Diameter: {D_o_orf*1e3:.2f} mm")

Fuel Diameter: 1.26 mm
Oxidiser Diameter: 1.20 mm


In [246]:
D_origin = 1.26e-3
D_shrinked = D_origin -0.25e-3

A_origin = np.pi * (D_origin/2)**2
A_shrink = np.pi * (D_shrinked/2)**2

Factor between these:

In [247]:
k = A_shrink / A_origin

print(f"Shrinkage Factor: {k}")

Shrinkage Factor: 0.6425422020660118


In [248]:
0.6*k

0.38552532123960703

## 8 - Generating our Output YAML File.

Finally, we can generate our output yaml file that can be read by downstream analysis. We firstly prepare our dictionaries to provide the necessary information for the dumps

In [244]:
fuel_dict = {
    "fuel": {
        "name": fu_injector["name"],
        "stiffness": float(performance["dp_fu/p_cc"]),
        "cda": float(geom["cda_fu"]),
    }
}

ox_dict = {
    "oxidiser":{
        "name": ox_injector["name"],
        "stiffness": float(performance["dp_ox/p_cc"]),
        "cda": float(geom["cda_ox"]),
    }
}

chamber_dict = {
    "chamber": {
        "a_t": float(geom["a_t"]),
        "a_e": float(geom["a_e"])
    }
}

m_dot_dict = {
    "m_dot":
        {
            "oxidiser": float(performance["m_dot_ox"]),
            "fuel": float(performance["m_dot_fu"]),
            "total": float(performance["m_dot"])
        }
}

gas_dict = {
    "gas":
        {
            "T": float(performance["T_o"]),
            "P": float(performance["gas"].p),
            "cp": float(performance["gas"].cp),
            "gamma": float(performance["gas"].gamma),
            "R": float(performance["gas"].R),
        }
}

MR_p = performance["m_dot_ox"] / performance["m_dot_fu"]

In [245]:
with open(f"{input_directory}/gg_out.yaml", "w") as f:
    
    # Hight Level Title
    f.write(f"#"*80 + "\n")
    f.write(f"#{"OT-2: Man-Ray and The Dirty Bubble":^78}#\n")
    f.write(f"#{f"{"":-^40}":^78}#\n")
    f.write(f"#{"Gas Generator Sizing Results":^78}#\n")
    f.write(f"#{"":^78}#\n")
    f.write(f"#{f"Generated at: XX:{datetime.now().minute:02}":^78}#\n")
    f.write(f"#{"":^78}#\n")
    f.write(f"#"*80 + "\n")
    f.write(f"\n\n")
    
    # Solution Convergence
    f.write(f"#"*80 + "\n")
    f.write(f"#{" Geometry ":-^78}#\n")
    f.write(f"#"*80 + "\n\n")
    
    f.write(f"gas_generator:\n\n")
    
    f.write(f"  geometry:\n\n")
    f.write(f"    injector:\n\n")
    fuel_str = yaml.dump(fuel_dict, default_flow_style=False)
    fuel_str = textwrap.indent(fuel_str, "      ")
    f.write(fuel_str)
    f.write(f"\n\n")
    
    oxidiser_str = yaml.dump(ox_dict, default_flow_style=False)
    oxidiser_str = textwrap.indent(oxidiser_str, "      ")
    f.write(oxidiser_str)
    f.write(f"\n\n")
    
    chamber_str = yaml.dump(chamber_dict, default_flow_style=False)
    chamber_str = textwrap.indent(chamber_str, "    ")
    f.write(chamber_str)
    f.write(f"\n\n")
    
    f.write(f"#"*80 + "\n")
    f.write(f"#{" Operating Point ":-^78}#\n")
    f.write(f"#"*80 + "\n\n")
    
    f.write(f"  operating_point:\n\n")
    
    f.write(f"    MR: {float(MR_p):.3f}\n\n")
    
    m_dot_str = yaml.dump(m_dot_dict, default_flow_style=False)
    m_dot_str = textwrap.indent(m_dot_str, "    ")
    f.write(m_dot_str)
    f.write(f"\n\n")
    
    # We can finally get the gas in
    gas_str = yaml.dump(gas_dict, default_flow_style=False)
    gas_str = textwrap.indent(gas_str, "    ")
    f.write(gas_str)
    f.write(f"\n\n")
    
